In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from tqdm import tqdm
import os

print("Loading data...")
# Corrected filename
df = pd.read_csv("../data/raw/complaints_raw.csv", low_memory=False)

print(f"Total rows: {df.shape[0]:,}")
print(f"Columns: {df.columns.tolist()}")

# Check product names
print("\nTop Products:")
print(df['Product'].value_counts().head(15))

# Relevant products (flexible matching)
relevant_products = ['Credit card', 'Personal loan', 'Savings account', 'Money transfer']

# Filter using contains (safer)
df_filtered = df[df['Product'].str.contains('|'.join(relevant_products), case=False, na=False)].copy()

print(f"\nFiltered rows (4 products): {len(df_filtered):,}")

# Narrative analysis
df_filtered['narrative'] = df_filtered['Consumer complaint narrative']
df_filtered['length'] = df_filtered['narrative'].str.len()

print("\nNarrative length statistics:")
print(df_filtered['length'].describe())

# Remove rows without narrative
df_clean = df_filtered[df_filtered['narrative'].notna() & 
                       (df_filtered['narrative'].str.strip() != '')].copy()

print(f"Final rows with narrative: {len(df_clean):,}")

# Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Remove common boilerplate
    text = re.sub(r'i am writing to file a complaint.*?', '', text, flags=re.IGNORECASE)
    text = re.sub(r'this is a complaint.*?', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[^a-z\s.,!?]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning text...")
tqdm.pandas()
df_clean['cleaned_narrative'] = df_clean['narrative'].progress_apply(clean_text)

# Save
os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/filtered_complaints.csv', index=False)
print("✅ Successfully saved: data/processed/filtered_complaints.csv")

Loading data...
Total rows: 9,609,797
Columns: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']

Top Products:
Product
Credit reporting or other personal consumer reports                             4834855
Credit reporting, credit repair services, or other personal consumer reports    2163857
Debt collection                                                                  799197
Mortgage                                                                         422254
Checking or savings account                                                      291178
Credit card                                                                      226686
Credit card or prepaid card                                                 

100%|██████████| 454472/454472 [02:19<00:00, 3247.92it/s]


✅ Successfully saved: data/processed/filtered_complaints.csv
